In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes torchcodec librosa peft evaluate jiwer

# Import & Data Loading

In [ ]:
import os, torch, gc, zipfile, glob
import pandas as pd
from datasets import Dataset, Audio
from huggingface_hub import hf_hub_download, login
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    BitsAndBytesConfig,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel  
)

BASE_DIR = "/kaggle/working/khotbah_dataset"
os.makedirs(BASE_DIR, exist_ok=True)

login("")

REPO_ID = "gracecalista/new-dataset-transcribe"
TOTAL_PARTS = 2

# Download Metadata
print("Mendownload metadata...")
csv_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="benchmarking_results_final.csv",
    repo_type="dataset",
    local_dir=BASE_DIR
)
df = pd.read_csv(csv_path).dropna(subset=['text', 'path'])

# Download & Extract semua part
for part_num in range(1, TOTAL_PARTS + 1):
    EXTRACT_FLAG = os.path.join(BASE_DIR, f".extracted_part{part_num}")
    zip_filename = f"wavs/wavs_part{part_num}.zip"

    if os.path.exists(EXTRACT_FLAG):
        with open(EXTRACT_FLAG, 'r') as f:
            flag_content = f.read()
        print(f"⏭️  Part {part_num}: Sudah diekstrak sebelumnya ({flag_content}), skip.")
        continue

    print(f"\n [{part_num}/{TOTAL_PARTS}] Mendownload {zip_filename}...")
    try:
        zip_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=zip_filename,
            repo_type="dataset",
            local_dir=BASE_DIR
        )
    except Exception as e:
        print(f"Part {part_num} gagal didownload: {e}")
        continue

    print(f"✅ Download selesai. Mengekstrak dari: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.infolist()
        total = len(members)
        print(f"Total file dalam zip: {total}")
        for i, member in enumerate(members, 1):
            zip_ref.extract(member, BASE_DIR)
            if i % 5000 == 0 or i == total:
                print(f"Ekstraksi: {i}/{total} file ({i/total*100:.1f}%)")

    extracted_wavs = glob.glob(f"{BASE_DIR}/**/*.wav", recursive=True)
    print(f"Part {part_num} selesai! Total .wav sejauh ini: {len(extracted_wavs)}")

    if len(extracted_wavs) == 0:
        print(f"ERROR: Tidak ada file .wav ditemukan setelah ekstraksi part {part_num}!")
    else:
        with open(EXTRACT_FLAG, 'w') as f:
            f.write(f"done={len(extracted_wavs)}")
        print(f"Flag part {part_num} disimpan.")

    os.remove(zip_path)
    print(f"Zip part {part_num} dihapus untuk hemat storage.")

# Mapping semua wav yang ada
all_wavs = glob.glob(f"{BASE_DIR}/**/*.wav", recursive=True)
if not all_wavs:
    print("ERROR: File .wav tidak ditemukan!")
else:
    path_map = {os.path.basename(p): p for p in all_wavs}
    df['audio_path'] = df['path'].apply(lambda x: path_map.get(os.path.basename(x)))
    df = df.dropna(subset=['audio_path'])

    print(f"\n JUMLAH DATA VALID: {len(df)}")
    if len(df) > 0:
        df = df.head(45000)
        ds = Dataset.from_dict({
            "audio": df["audio_path"].tolist(),
            "sentence": df["text"].tolist()
        })
        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        print("Dataset BERHASIL dibuat di Kaggle!")

# Preprocessing

In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = text.replace("1", "satu")
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = ds.map(lambda x: {"sentence": normalize_text(x["sentence"])})

## Model Feature Extractor

In [ ]:
MODEL_NAME = "openai/whisper-large-v3"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # Feature extraction (Spectrogram)
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Tokenisasi teks dengan truncation
    batch["labels"] = tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=448  # Batas maksimal Whisper
    ).input_ids

    return batch

# Filter teks yang kosong secara langsung dari string-nya
ds = ds.filter(lambda x: len(x["sentence"].strip()) > 0)

# Shuffle + split
ds = ds.shuffle(seed=42).train_test_split(test_size=0.1)

# Model Setup

In [ ]:
# Load Processor, Tokenizer
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="indonesian", task="transcribe")
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="indonesian", task="transcribe")

# Konfigurasi 4-Bit 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Muat Model dengan quantization
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Persiapkan untuk kbit training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False  # Dihandle oleh TrainingArguments
)

# Konfigurasi LoRA 
peft_config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    r=8,           
    lora_alpha=16, 
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, peft_config)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="indonesian",
    task="transcribe"
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="indonesian",
    task="transcribe"
)
model.print_trainable_parameters()

# Config Stabilitas
model.enable_input_require_grads()  
model.config.use_cache = False      

# Trainer

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    print("\nDEBUG")
    print("PRED:", pred_str[:2])
    print("LABEL:", label_str[:2])

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# untuk lanjut fine-tuning (kalau mulai dari awal, tidak perlu di-run)
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="gracecalista/whisper-khotbah-lora-v3",
    local_dir="./whisper-khotbah-lora-v3",
)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import EarlyStoppingCallback

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # 1. Ekstraksi Audio (Spectrogram) ON-THE-FLY 
        input_features = []
        for f in features:
            extracted = self.processor.feature_extractor(
                f["audio"]["array"],
                sampling_rate=f["audio"]["sampling_rate"]
            ).input_features[0]
            input_features.append({"input_features": extracted})

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. Tokenisasi Label Teks ON-THE-FLY
        label_features = []
        for f in features:
            tokenized = self.processor.tokenizer(
                f["sentence"],
                truncation=True,
                max_length=448
            ).input_ids
            label_features.append({"input_ids": tokenized})

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 3. Ganti padding token ID dengan -100 agar diabaikan oleh loss function
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch


class MemoryCleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()


training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-khotbah-lora-v3",

    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    num_train_epochs=3,

    learning_rate=2e-5,
    warmup_steps=100,

    fp16=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    predict_with_generate=True,
    generation_max_length=225,

    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    remove_unused_columns=False,
    report_to="none",

    push_to_hub=True,
    hub_model_id="gracecalista/whisper-khotbah-lora-v3",
    hub_strategy="all_checkpoints",

    max_grad_norm=1.0
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=ds["train"],
    eval_dataset=ds["test"],

    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),

    processing_class=processor.feature_extractor,

    compute_metrics=compute_metrics,

    callbacks=[MemoryCleanupCallback(), EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🚀 Training dimulai...")
trainer.train() #atau trainer.train(resume_from_checkpoint="./whisper-khotbah-lora-v3/checkpoint-800") --> menyesuaikan checkpoint terakhir
trainer.push_to_hub()

In [ ]:
# gc.collect()
# torch.cuda.empty_cache()